# ISCXIDS2012 — Dataset Exploration

Multi-day network capture with labelled XML flow files.

**File:** `iscxids2012-master.tar.gz` — must be extracted before use.

> **Run cell 2 once** to extract the archive, then proceed.

In [ ]:
import tarfile, os
from pathlib import Path

BASE    = Path('../../dataset/iscxids2012/')
ARCHIVE = BASE / 'iscxids2012-master.tar.gz'

if ARCHIVE.exists() and not any(BASE.glob('*.xml')):
    print('Extracting archive...')
    with tarfile.open(ARCHIVE) as tar:
        tar.extractall(BASE)
    print('Done.')
else:
    print('Already extracted or archive not found.')

xml_files = sorted(BASE.rglob('*.xml'))
print(f'XML files found: {len(xml_files)}')
for x in xml_files: print(f'  {x.name}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')

## 1 · Parse XML Flow Files

In [ ]:
def parse_xml_flows(xml_path):
    """Parse one ISCX labelled-flow XML into a DataFrame."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    records = []
    for flow in root.iter('Flow'):          # tag may vary; inspect root if empty
        rec = {child.tag: child.text for child in flow}
        records.append(rec)
    return pd.DataFrame(records)

if xml_files:
    sample_df = parse_xml_flows(xml_files[0])
    print(f'Columns ({len(sample_df.columns)}): {list(sample_df.columns)}')
    print(f'Rows in first file: {len(sample_df)}')
    sample_df.head(3)
else:
    print('No XML files found — check extraction path above.')

## 2 · Load All Days

In [ ]:
dfs = []
for f in xml_files:
    tmp = parse_xml_flows(f)
    tmp['source_file'] = f.stem
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)
print(f'Combined shape: {df.shape}')
df.head(3)

## 3 · Data Types — XML Loads Everything as String

In [ ]:
# All columns come in as object; identify which should be numeric
print('dtypes:', df.dtypes.value_counts().to_dict())

# Attempt numeric conversion on all columns
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='ignore')

print('After conversion:', df.dtypes.value_counts().to_dict())
print()
print('Remaining object columns:', df.select_dtypes('object').columns.tolist())

## 4 · Label Column

In [ ]:
# The label column is usually 'Tag', 'label', 'appName', or similar — inspect
obj_cols = df.select_dtypes('object').columns.tolist()
for col in obj_cols:
    print(f'{col}: {df[col].nunique()} unique → {df[col].unique()[:6]}')

In [ ]:
# Adjust 'label_col' to the correct column name found above
label_col = 'Tag'   # <-- update if different

if label_col in df.columns:
    print(df[label_col].value_counts().to_string())

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    df[label_col].value_counts().plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_title('Multi-class distribution')
    df[label_col].apply(lambda x: 'Normal' if 'normal' in str(x).lower() else 'Attack') \
        .value_counts().plot(kind='bar', ax=axes[1], color=['steelblue','tomato'])
    axes[1].set_title('Binary split'); axes[1].tick_params(axis='x', rotation=0)
    plt.tight_layout(); plt.show()
else:
    print(f'Column "{label_col}" not found. Check obj_cols above and update label_col.')

## 5 · Missing Values

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0].to_string() or 'No missing values')

## 6 · Feature Statistics

In [ ]:
num_cols = df.select_dtypes('number').columns
df[num_cols].describe().T.round(3)

## 7 · Preprocessing Checklist

| Step | Required | Notes |
|---|---|---|
| Extract tar.gz | **Done above** | One-time step |
| XML → DataFrame | **Done above** | All values parse as string initially |
| Numeric coercion | Yes | `pd.to_numeric(..., errors='ignore')` on all columns |
| Identify label column | Yes | Likely `Tag` or `appName` — verify from output above |
| Drop metadata columns | Yes | IP addresses, timestamps, source_file |
| Encode categoricals | Yes | Protocol, any remaining string columns |
| Scale numerics | Yes | Duration/byte features are skewed |
| Class imbalance | Yes | Normal traffic likely dominates |

## 8 · Train / Test Split

In [ ]:
from sklearn.preprocessing import LabelEncoder

if label_col in df.columns:
    # Drop non-feature columns (update list as needed after inspecting obj_cols)
    drop_cols = ['source_file'] + [c for c in obj_cols if c != label_col]
    df_model = df.drop(columns=[c for c in drop_cols if c in df.columns])

    le = LabelEncoder()
    y  = le.fit_transform(df_model[label_col])
    X  = df_model.drop(columns=[label_col])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
    print(f'Classes: {list(le.classes_)}')
else:
    print('Update label_col before running the split.')

## 9 · Summary

- **Format:** XML flow files (one per capture day) — needs parsing step
- **Features:** Network flow attributes; all initially strings, coerce to numeric
- **Labels:** Normal + HttpDoS, DDoS, BruteForce-SSH, Infiltration (exact names vary)
- **Key issues:** XML parsing, dtype coercion, identifying the correct label column
- **Smallest dataset:** fewer unique attack scenarios than 2017/2018 datasets